# Tarea 2 · Parte 1 — Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Pueden convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| | |
|---|---|
| **Fecha de extracción** | Documentos: 23/10/2026 · Audio: 24/10/2026 |
| **Entorno** | VS Code (Jupyter) en laptop, CPU, Python 3.12 — mismas librerías que los grupos `docai` y `audio` del curso (ver celda de instalación) |
| **OCR** | PaddleOCR 3.7.0 (`lang="es"`, configuración base, sin personalización) — Sesión 10 |
| **ASR** | faster-whisper large-v3-turbo (`mobiuslabsgmbh/faster-whisper-large-v3-turbo`, VAD activado) con `voz.py` de la Sesión 13 |
| **Diarización** | Huellas de voz WeSpeaker ResNet34 (`pyannote/wespeaker-voxceleb-resnet34-LM`) + clustering jerárquico (`voz.diarizar`) |

Las versiones exactas instaladas se imprimen en las celdas de configuración de cada parte.

### Corpus A · Documentos (25)

| Tipo | n | Fuente | Licencia |
|---|---|---|---|
| Impresos nítidos | 8 | Wikimedia Commons y Repositorio PUCP (*El Peruano* 1844, *Mercurio Peruano* 1791, *El Comercio* 1839/c.1842/1945, billete de 1879, *La Mesa Peruana* 1867, Constitución de 1823) | Dominio público / CC0 |
| Fotos con sombra, inclinación o poca luz | 8 | Fotografías propias (Jesús María, Lima) | CC BY 4.0 (propia) |
| Manuscritos | 9 | Fotografías propias de pizarras de menú y apuntes | CC BY 4.0 (propia) |

URL, autor y licencia de cada archivo: `datos/fuentes.csv`.

### Corpus B · Audio (≥ 30 min, ≥ 20 clips)

| Fuente | Contenido | Licencia |
|---|---|---|
| [OpenSLR 73](https://www.openslr.org/73/) (Google) | Español peruano leído, hablantes anónimos | CC BY-SA 4.0 |
| [Wikimedia Commons](https://commons.wikimedia.org) | Entrevistas en la calle en El Agustino (Lima) y a una dirigente de Fenmucarinap | CC BY-SA 4.0 |

Al menos un audio tiene 2 o más hablantes (entrevista 2 de El Agustino y un diálogo armado con dos hablantes de OpenSLR 73) para la diarización.

### Ética y datos personales
- Solo archivos con licencia libre; atribución completa en `datos/fuentes.csv`.
- Los originales (imágenes y audio) están en `datos/raw/`, que **no se versiona** (`.gitignore`).
- La lista de nombres reales a seudonimizar está en un archivo privado (`datos/raw/reemplazos_privados.csv`) que tampoco se sube.
- `doc_012` y `doc_014` no se publican como imagen (personas identificables / datos de un particular); solo su texto seudonimizado.

### Cómo ejecutar este notebook
1. Abrir la carpeta `tarea_2/` en VS Code y crear el entorno (una sola vez, en la terminal):
   ```
   py -3.12 -m venv .venv
   .venv\Scripts\activate
   pip install "paddlepaddle>=3.2.1" "paddleocr==3.7.0" faster-whisper jiwer rapidfuzz "pyannote.audio>=4.0.7" soundfile scipy pandas requests plotly kaleido ipykernel
   ```
2. En el notebook, elegir el kernel `.venv` (arriba a la derecha) y ejecutar de arriba hacia abajo.
3. Cada parte (A: OCR, B: audio) relee de disco lo que necesita, así que se puede reiniciar el kernel entre partes si falta memoria.

---
# Parte A · OCR de documentos (Sesión 10)

In [1]:
# Rutas: el notebook está en tarea_2/ y lee las imágenes originales de datos/raw/documentos (no versionado)
from pathlib import Path
BASE = Path.cwd()
assert (BASE / "datos").exists(), "Abre el notebook desde la carpeta tarea_2"
DOCS = BASE / "datos/raw/documentos"
print("Imágenes encontradas:", len([p for p in DOCS.glob("doc_*.*") if p.is_file()]))

Imágenes encontradas: 25


In [ ]:
import torch            # en Windows, torch debe cargarse ANTES que paddle

In [3]:
   import numpy as np
   import pandas as pd
   import torch            # en Windows, torch debe cargarse ANTES que paddle
   import paddle, paddleocr
   from paddleocr import PaddleOCR

c:\Users\valer\Documents\Github\python_intermedio\tarea_2\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
c:\Users\valer\Documents\Github\python_intermedio\tarea_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import hashlib, json, os, time, warnings, logging
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
warnings.filterwarnings("ignore")
for nombre in ("paddlex", "paddleocr", "ppocr"):
    logging.getLogger(nombre).setLevel(logging.ERROR)

from PIL import Image, ImageOps

pd.set_option("display.max_colwidth", 80)
import platform
print("Python", platform.python_version(), "· PaddleOCR", paddleocr.__version__, "· PaddlePaddle", paddle.__version__,
      "· GPU:", paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0)

Python 3.12.15 · PaddleOCR 3.7.0 · PaddlePaddle 3.3.1 · GPU: False


### A.1 Inventario y duplicados por hash

El tipo de cada documento y su fuente vienen de `datos/raw/fuentes_documentos.csv` (la tabla que llené al recolectar).
Dos archivos con el mismo hash SHA-256 son el mismo archivo: se descarta el segundo.

In [7]:
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

meta = pd.read_csv(BASE / "datos/raw/fuentes_documentos.csv", dtype=str, sep=None, engine="python", encoding="utf-8-sig").fillna("")
meta.columns = meta.columns.str.strip()
archivos = {p.stem: p for p in sorted(DOCS.glob("doc_*.*")) if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}}

inv = meta[meta["id"].isin(archivos)].copy()
inv["ruta"] = inv["id"].map(archivos)
inv["sha256"] = inv["ruta"].map(sha256)
duplicados = inv[inv.duplicated("sha256", keep="first")]
inv = inv.drop(duplicados.index)

sin_archivo = sorted(set(meta["id"]) - set(archivos))
print(f"Documentos en la tabla: {len(meta)} · con imagen: {len(inv) + len(duplicados)}")
print(f"Duplicados por hash descartados: {len(duplicados)} {list(duplicados['id'])}")
print(f"En la tabla pero sin imagen: {sin_archivo}")
inv["tipo"].value_counts()

Documentos en la tabla: 25 · con imagen: 25
Duplicados por hash descartados: 0 []
En la tabla pero sin imagen: []


tipo
manuscrito    9
impreso       8
foto          8
Name: count, dtype: int64

### A.2 Preprocesamiento

Parámetros (iguales para todos los documentos, fijados antes de ver resultados):
- Se corrige la rotación guardada en los metadatos EXIF de las fotos del celular (`ImageOps.exif_transpose`).
- Se convierte a RGB y, si el lado mayor pasa de 2400 px, se reduce a 2400 px (PaddleOCR redimensiona internamente; esto solo ahorra memoria).
- No se binariza ni se endereza: la Parte 2 debe medir cómo lee el OCR **base** las fotos inclinadas y con sombra.

La copia preprocesada queda en `datos/raw/documentos/proc/`.

In [8]:
PROC = DOCS / "proc"; PROC.mkdir(exist_ok=True)
LADO_MAX = 2400

def preparar(ruta):
    im = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    if max(im.size) > LADO_MAX:
        im.thumbnail((LADO_MAX, LADO_MAX))
    destino = PROC / f"{ruta.stem}.png"
    im.save(destino)
    return destino, im.size

inv[["ruta_proc", "tamano"]] = inv["ruta"].apply(lambda r: pd.Series(preparar(r)))
inv[["id", "tipo", "tamano"]].head()

,id,tipo,tamano
0,doc_001,impreso,"(960, 1360)"
1,doc_002,impreso,"(694, 304)"
2,doc_003,impreso,"(960, 745)"
3,doc_004,impreso,"(960, 1448)"
4,doc_005,impreso,"(500, 722)"


### A.3 OCR con PaddleOCR

Configuración base: `lang="es"`, sin clasificador de orientación de página, sin "planchado" de hojas y sin orientación por línea
(las mismas opciones de la clase). Para cada documento se guarda el **respaldo crudo** (JSON con texto, confianza y caja de cada línea),
la imagen con las cajas dibujadas y los **recortes** de cada línea en `datos/raw/documentos/`.

In [14]:
# enable_mkldnn=False: en Windows/CPU, PaddlePaddle 3.3 con oneDNN falla
OPCIONES_OCR = dict(lang="es", use_doc_orientation_classify=False, use_doc_unwarping=False,
                    use_textline_orientation=False, enable_mkldnn=False)
ocr = PaddleOCR(**OPCIONES_OCR)
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (lang=es, config. base, CPU sin oneDNN)"

CRUDO = DOCS / "ocr_crudo"; CRUDO.mkdir(exist_ok=True)
CAJAS = DOCS / "ocr_cajas"; CAJAS.mkdir(exist_ok=True)
RECORTES = DOCS / "recortes"; RECORTES.mkdir(exist_ok=True)

filas = []
for d in inv.itertuples():
    t0 = time.perf_counter()
    r = ocr.predict(str(d.ruta_proc))[0]
    seg = time.perf_counter() - t0
    textos, confs = list(r["rec_texts"]), [float(c) for c in r["rec_scores"]]
    cajas = np.asarray(r["rec_boxes"]).reshape(-1, 4) if len(textos) else np.zeros((0, 4))

    try:                                    # respaldo crudo (JSON) e imagen con las cajas
        r.save_to_json(str(CRUDO / f"{d.id}.json"))
        r.save_to_img(str(CAJAS / d.id))
    except Exception as e:
        print("   (no se pudo guardar el respaldo:", e, ")")
    im = Image.open(d.ruta_proc)
    carpeta = RECORTES / d.id; carpeta.mkdir(exist_ok=True)
    for k, (x0, y0, x1, y1) in enumerate(cajas.astype(int)):
        im.crop((x0, y0, x1, y1)).save(carpeta / f"linea_{k:03d}.png")

    filas.append({"id": d.id, "texto_ocr_crudo": "\n".join(textos), "n_lineas": len(textos),
                  "confianza_media": round(float(np.mean(confs)), 3) if confs else 0.0,
                  "segundos_ocr": round(seg, 2)})
    print(f"{d.id} [{d.tipo:<10}] {len(textos):3d} líneas · conf. {filas[-1]['confianza_media']:.2f} · {seg:4.1f} s")

ocr_df = inv.merge(pd.DataFrame(filas), on="id")
ocr_df["modelo_ocr"] = MODELO_OCR

Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/fonts/simfang.ttf ...
[==================================================] 100.00%


doc_001 [impreso   ] 138 líneas · conf. 0.97 · 234.2 s
doc_002 [impreso   ]  15 líneas · conf. 0.94 · 15.4 s
doc_003 [impreso   ]  47 líneas · conf. 0.91 · 85.7 s
doc_004 [impreso   ] 286 líneas · conf. 0.93 · 423.0 s
doc_005 [impreso   ] 157 líneas · conf. 0.67 · 225.4 s
doc_006 [impreso   ] 122 líneas · conf. 0.96 · 177.8 s
doc_007 [impreso   ]  23 líneas · conf. 0.99 · 58.8 s
doc_008 [impreso   ]  15 líneas · conf. 0.94 · 24.7 s
doc_009 [foto      ]  28 líneas · conf. 0.98 · 22.7 s
doc_010 [foto      ] 116 líneas · conf. 0.98 · 140.5 s
doc_011 [foto      ] 104 líneas · conf. 0.89 · 114.2 s
doc_012 [foto      ]  36 líneas · conf. 0.99 · 37.9 s
doc_013 [foto      ] 150 líneas · conf. 0.94 · 158.8 s
doc_014 [foto      ]  19 líneas · conf. 0.99 · 24.0 s
doc_015 [foto      ]  17 líneas · conf. 0.95 · 41.1 s
doc_016 [foto      ]  14 líneas · conf. 0.99 · 51.2 s
doc_017 [manuscrito]  42 líneas · conf. 0.95 · 64.2 s
doc_018 [manuscrito]   7 líneas · conf. 0.94 · 20.1 s
doc_019 [manuscrito] 

In [15]:
# Se guarda en datos/raw/ocr_crudo.csv (local, no versionado) para que la Parte B pueda releerlo

cols = ["id", "tipo", "descripcion", "fuente_url", "licencia", "autor", "acceso", "publicar_imagen",
        "sha256", "texto_ocr_crudo", "n_lineas", "confianza_media", "segundos_ocr", "modelo_ocr"]
ocr_df["archivo"] = "datos/raw/documentos/" + ocr_df["ruta"].map(lambda p: p.name)
ocr_df[cols + ["archivo"]].to_csv(BASE / "datos/raw/ocr_crudo.csv", index=False)